# Longrun ozone-loss predictability from stratospheric dynamics

## Research Question

**How early can winter stratospheric dynamics identify and quantify severe Arctic ozone-loss years in the longrun?**

This notebook treats predictability as two linked questions, not just one ML score. In plain terms, the answer is: **not only** whether a year enters the low-25% O3 class, and **not** a full simulation of the daily O3 trajectory. It is an early statistical forecast, using physically interpretable winter dynamics, of both severe-event membership and approximate severity:

1. **Event predictability**: can the model issue an early alarm that a year will enter the lowest 25% of March-April polar-cap ozone?  This is the `Low25_min_label` classification problem.
2. **Intensity predictability**: can the model estimate how severe the ozone loss will be?  This is evaluated two ways:
   - March-April mean ozone-loss anomaly, `MA_O3_loss`, which is the smoother seasonal loss target.
   - March-April minimum ozone depth, `MA_O3_min_DU`, which directly asks how low the year roughly gets.

Operationally, the **stable usable lead time** is defined from the B2000WCN out-of-fold test. A cutoff is called usable when it satisfies all event-alarm and seasonal-loss criteria:

- event AUC >= 0.75,
- Brier skill versus climatology > 0,
- top-quartile alarm hit rate >= 0.60,
- top-quartile false-alarm rate <= 0.35,
- `MA_O3_loss` regression correlation >= 0.50,
- `MA_O3_loss` regression MSE skill > 0.

The annual-minimum target, `MA_O3_min_DU`, is reported separately as an intensity diagnostic because a single-day minimum is noisier than the seasonal loss. It answers whether we can estimate the depth of the event, but it is not used alone to declare a stable forecast window.

## Physical Framing

The predictors are split by stratospheric dynamics:

- `VORTEX_STATE`: NAM 10/50 hPa, representing vortex strength and vertical coupling.
- `WAVE_FORCING`: EP flux and EP-flux divergence, representing planetary-wave forcing of the vortex.
- `DYN_COUPLED`: vortex state plus wave forcing.
- `DYN_PLUS_PASSIVE_TRACER`: dynamics plus early O3 anomaly as a passive tracer of prior transport/chemistry. O3 is not evaluated as a standalone non-dynamical predictor set.


In [1]:

from __future__ import annotations

import json
import math
import os
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import pearsonr
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.metrics import brier_score_loss, mean_squared_error, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "ML").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside code_cleaned.")

REPO_ROOT = find_repo_root()
OUT_DIR = REPO_ROOT / "ML" / "outputs" / "dynamical_predictability"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH_LENGTHS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
MONTH_START_DOY0 = np.concatenate([[0], np.cumsum(MONTH_LENGTHS)[:-1]])

TARGET_START_DOY = 60
CUTOFF_DOYS = list(range(1, 60))
SHORT_WINDOW_DAYS = 28
LONG_WINDOW_DAYS = 90
EVENT_FRACTION = 0.25
RANDOM_STATE = 42

print("repo:", REPO_ROOT)
print("outputs:", OUT_DIR)


repo: /home/weiji/restart_exam/code_cleaned
outputs: /home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability


## Dataset paths

Inputs are read from the cleaned longrun products under `/mnt/soclim0/public_data/weiji`.  The notebook writes only inside `code_cleaned/ML`.


In [2]:

@dataclass(frozen=True)
class DatasetConfig:
    name: str
    o3_file: Path
    nam_file: Path
    ep_file: Path

DATASETS: Mapping[str, DatasetConfig] = {
    "B2000WCN001002": DatasetConfig(
        name="B2000WCN001002",
        o3_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/partial_O3/B2000WCN_partial_O3_all_ranges.nc"),
        nam_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/NAM/B2000WCN001002_Vertical_NAM.nc"),
        ep_file=Path("/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/EPflux_daily_ubar_wcorr/all_waves/EPFLUX_all_waves_210yr_time_plev_lat.nc"),
    ),
    "BWCN": DatasetConfig(
        name="BWCN",
        o3_file=Path("/mnt/soclim0/public_data/weiji/BWCN/partial_O3/BWCN_partial_O3_all_ranges.nc"),
        nam_file=Path("/mnt/soclim0/public_data/weiji/BWCN/NAM/BWCN_Vertical_NAM.nc"),
        ep_file=Path("/mnt/soclim0/public_data/weiji/BWCN/EPflux_daily_ubar_wcorr/all_waves/EPFLUX_all_waves_24yr_time_plev_lat.nc"),
    ),
}

def require_inputs(configs: Iterable[DatasetConfig]) -> None:
    missing = []
    for cfg in configs:
        for path in [cfg.o3_file, cfg.nam_file, cfg.ep_file]:
            if not path.exists():
                missing.append(path)
    if missing:
        raise FileNotFoundError("Missing inputs:\n" + "\n".join(str(p) for p in missing))

require_inputs(DATASETS.values())
print("Input files found.")


Input files found.


## Helper functions

The functions below keep the time handling explicit. The longrun products are no-leap 365-day years; where CAM `date` exists, it is used directly.


In [3]:

def doy_from_month_day(month: np.ndarray, day: np.ndarray) -> np.ndarray:
    return MONTH_START_DOY0[month.astype(int) - 1] + day.astype(int)

def month_day_from_doy(doy: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    doy0 = doy.astype(int) - 1
    month = np.searchsorted(np.cumsum(MONTH_LENGTHS), doy0, side="right") + 1
    day = doy0 - MONTH_START_DOY0[month - 1] + 1
    return month.astype(int), day.astype(int)

def month_day_label(month: Sequence[int], day: Sequence[int]) -> List[str]:
    return [f"{int(m):02d}-{int(d):02d}" for m, d in zip(month, day)]

def frame_from_numeric_time(n_time: int, year_start: int = 1) -> pd.DataFrame:
    idx = np.arange(n_time, dtype=int)
    year = idx // 365 + year_start
    doy = idx % 365 + 1
    month, day = month_day_from_doy(doy)
    return pd.DataFrame({
        "year": year.astype(int),
        "doy": doy.astype(int),
        "month": month,
        "day": day,
        "month_day": month_day_label(month, day),
        "abs_day": ((year - 1) * 365 + doy).astype(int),
    })

def frame_from_cam_date(date_values: np.ndarray) -> pd.DataFrame:
    date = date_values.astype(int)
    year = date // 10000
    month = (date % 10000) // 100
    day = date % 100
    doy = doy_from_month_day(month, day)
    return pd.DataFrame({
        "year": year.astype(int),
        "doy": doy.astype(int),
        "month": month.astype(int),
        "day": day.astype(int),
        "month_day": month_day_label(month, day),
        "abs_day": ((year - 1) * 365 + doy).astype(int),
    })

def detect_data_var(ds: xr.Dataset, candidates: Sequence[str]) -> str:
    for name in candidates:
        if name in ds.data_vars:
            return name
    numeric = [name for name, da in ds.data_vars.items() if da.ndim >= 1 and np.issubdtype(da.dtype, np.number)]
    if not numeric:
        raise ValueError(f"No numeric data variable found: {list(ds.data_vars)}")
    return numeric[0]

def select_pressure(da: xr.DataArray, coord_name: str, target_hpa: float) -> xr.DataArray:
    values = np.asarray(da[coord_name].values, dtype=float)
    target = target_hpa * 100.0 if np.nanmax(values) > 2000.0 else target_hpa
    nearest = float(values[np.nanargmin(np.abs(values - target))])
    return da.sel({coord_name: nearest})

def linear_trend(values: np.ndarray, x: np.ndarray) -> float:
    mask = np.isfinite(values) & np.isfinite(x)
    if mask.sum() < 5:
        return np.nan
    xx = x[mask].astype(float)
    yy = values[mask].astype(float)
    xx = xx - xx.mean()
    return float(np.polyfit(xx, yy, 1)[0])

def window_mean(df: pd.DataFrame, value_col: str, start_abs: int, end_abs: int, min_count: int) -> float:
    sub = df[(df["abs_day"] >= start_abs) & (df["abs_day"] <= end_abs)]
    values = sub[value_col].astype(float)
    if values.notna().sum() < min_count:
        return np.nan
    return float(values.mean())

def window_trend(df: pd.DataFrame, value_col: str, start_abs: int, end_abs: int, min_count: int) -> float:
    sub = df[(df["abs_day"] >= start_abs) & (df["abs_day"] <= end_abs)]
    values = sub[value_col].astype(float)
    if values.notna().sum() < min_count:
        return np.nan
    return linear_trend(values.to_numpy(), sub["abs_day"].to_numpy())

def cutoff_label(cutoff_doy: int) -> str:
    month, day = month_day_from_doy(np.array([cutoff_doy]))
    return f"{int(month[0]):02d}-{int(day[0]):02d}"


## Load physically interpretable daily indices

- O3 is used to define the target, and later only as a passive-tracer predictor inside the combined dynamical model.
- NAM at 10/30/50 hPa is used as a compact vortex-strength and vertical-coupling description.
- EP flux and EP-flux divergence are used as wave-activity forcing proxies.


In [4]:

def load_o3_daily(cfg: DatasetConfig) -> pd.DataFrame:
    print(f"[load] {cfg.name}: O3 target/passive tracer", flush=True)
    with xr.open_dataset(cfg.o3_file, decode_times=False) as ds:
        var = detect_data_var(ds, ["O3_partial_60_90N_30_70hPa"])
        values = ds[var].astype("float64").load().values
        frame = frame_from_cam_date(ds["date"].load().values) if "date" in ds else frame_from_numeric_time(values.shape[0])
    frame["O3_30_70_60_90N_DU"] = values
    frame["O3_rm5"] = pd.Series(values).rolling(5, center=True, min_periods=3).mean().to_numpy()
    frame["O3_clim_rm5"] = frame.groupby("month_day")["O3_rm5"].transform("mean")
    frame["O3_anom_rm5"] = frame["O3_rm5"] - frame["O3_clim_rm5"]
    return frame

def load_nam_daily(cfg: DatasetConfig, levels_hpa=(10, 30, 50)) -> pd.DataFrame:
    print(f"[load] {cfg.name}: NAM vortex-state levels {levels_hpa}", flush=True)
    with xr.open_dataset(cfg.nam_file, decode_times=False) as ds:
        var = detect_data_var(ds, ["NAM_Vertical", "NAM", "nam_vertical", "nam"])
        da = ds[var]
        lev_name = "lev" if "lev" in da.coords else list(da.dims)[-1]
        columns = {}
        for level in levels_hpa:
            columns[f"NAM{level}"] = select_pressure(da, lev_name, float(level)).astype("float64").load().values
    frame = frame_from_numeric_time(len(next(iter(columns.values()))))
    for name, values in columns.items():
        frame[name] = values
    frame["NAM10_minus_NAM50"] = frame["NAM10"] - frame["NAM50"]
    frame["NAM30_minus_NAM50"] = frame["NAM30"] - frame["NAM50"]
    return frame

def load_ep_daily(cfg: DatasetConfig, levels_hpa=(50, 100)) -> pd.DataFrame:
    print(f"[load] {cfg.name}: EP flux wave forcing levels {levels_hpa}", flush=True)
    columns = {}
    with xr.open_dataset(cfg.ep_file, decode_times=False) as ds:
        for var in ["ep2", "div2"]:
            if var not in ds:
                continue
            da = ds[var]
            for level in levels_hpa:
                selected = select_pressure(da, "plev", float(level))
                if "lat" in selected.dims:
                    sub = selected.sel(lat=slice(40, 80))
                    if sub.sizes.get("lat", 0) == 0:
                        sub = selected.sel(lat=slice(80, 40))
                    weights = np.cos(np.deg2rad(sub["lat"]))
                    selected = sub.weighted(weights).mean("lat")
                columns[f"{var.upper()}_{level}hPa_40_80N"] = selected.astype("float64").load().values
    frame = frame_from_numeric_time(len(next(iter(columns.values()))))
    for name, values in columns.items():
        frame[name] = values
    return frame

def build_target_table(o3_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    rows = []
    for year, group in o3_df.groupby("year"):
        ma = group[group["month"].isin([3, 4])].copy()
        if ma["O3_rm5"].notna().sum() < 50:
            continue
        min_idx = ma["O3_rm5"].idxmin()
        rows.append({
            "dataset": dataset_name,
            "year": int(year),
            "MA_O3_anom": float(ma["O3_anom_rm5"].mean()),
            "MA_O3_loss": float(-ma["O3_anom_rm5"].mean()),
            "MA_O3_min_DU": float(ma["O3_rm5"].min()),
            "MA_O3_min_doy": int(ma.loc[min_idx, "doy"]),
        })
    target = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    n_event = max(1, int(math.floor(EVENT_FRACTION * len(target))))
    target["low_o3_rank"] = target["MA_O3_min_DU"].rank(method="first", ascending=True)
    target["Low25_min_label"] = (target["low_o3_rank"] <= n_event).astype(int)
    target["n_event"] = n_event
    return target

def load_bundle(cfg: DatasetConfig) -> Dict[str, pd.DataFrame]:
    o3 = load_o3_daily(cfg)
    nam = load_nam_daily(cfg)
    ep = load_ep_daily(cfg)
    target = build_target_table(o3, cfg.name)
    print(f"[target] {cfg.name}: n_years={len(target)}, n_events={int(target['Low25_min_label'].sum())}", flush=True)
    return {"o3": o3, "nam": nam, "ep": ep, "target": target}

bundles = {name: load_bundle(cfg) for name, cfg in DATASETS.items()}
bundles["B2000WCN001002"]["target"].head()


[load] B2000WCN001002: O3 target/passive tracer


[load] B2000WCN001002: NAM vortex-state levels (10, 30, 50)


[load] B2000WCN001002: EP flux wave forcing levels (50, 100)


[target] B2000WCN001002: n_years=207, n_events=51


[load] BWCN: O3 target/passive tracer


[load] BWCN: NAM vortex-state levels (10, 30, 50)


[load] BWCN: EP flux wave forcing levels (50, 100)


[target] BWCN: n_years=23, n_events=5


,dataset,year,MA_O3_anom,MA_O3_loss,MA_O3_min_DU,MA_O3_min_doy,low_o3_rank,Low25_min_label,n_event
0,B2000WCN001002,1,-2.405034,2.405034,98.700822,120,71.0,0,51
1,B2000WCN001002,2,-8.793734,8.793734,94.816771,90,38.0,1,51
2,B2000WCN001002,3,-20.850690,20.850690,83.638120,88,5.0,1,51
3,B2000WCN001002,4,-5.104077,5.104077,97.625508,97,56.0,0,51
4,B2000WCN001002,5,-0.410272,0.410272,98.540256,120,68.0,0,51


## Feature groups

The split is meant to ask physical questions:

1. Does the vortex state alone contain enough information?
2. Does wave forcing alone contain enough information?
3. Does the vortex-wave coupled state produce a longer stable lead time?
4. Does adding O3 as a passive tracer of prior transport/chemistry sharpen the alarm without changing the dynamic framing?


In [5]:

PREDICTOR_SETS: Mapping[str, List[str]] = OrderedDict([
    ("VORTEX_STATE", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
    ]),
    ("WAVE_FORCING", [
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
    ]),
    ("DYN_COUPLED", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
    ]),
    ("DYN_PLUS_PASSIVE_TRACER", [
        "NAM10_short_mean",
        "NAM50_short_mean",
        "NAM50_short_trend",
        "NAM10_minus_NAM50_short_mean",
        "EP2_100hPa_long_mean",
        "EP2_50hPa_long_mean",
        "DIV2_100hPa_long_mean",
        "EP2_100hPa_short_mean",
        "EP2_100hPa_long_trend",
        "O3_short_mean",
        "O3_short_trend",
    ]),
])

def build_features_for_cutoff(cutoff_doy: int, bundle: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    target = bundle["target"]
    o3 = bundle["o3"]
    nam = bundle["nam"]
    ep = bundle["ep"]
    rows = []
    for _, tr in target.iterrows():
        year = int(tr["year"])
        cutoff_abs = (year - 1) * 365 + cutoff_doy
        short_start = cutoff_abs - SHORT_WINDOW_DAYS + 1
        long_start = cutoff_abs - LONG_WINDOW_DAYS + 1
        row = {
            "dataset": tr["dataset"],
            "year": year,
            "cutoff_doy": int(cutoff_doy),
            "cutoff_label": cutoff_label(cutoff_doy),
            "lead_days_to_Mar1": int(TARGET_START_DOY - cutoff_doy),
            "MA_O3_loss": float(tr["MA_O3_loss"]),
            "MA_O3_min_DU": float(tr["MA_O3_min_DU"]),
            "MA_O3_min_doy": int(tr["MA_O3_min_doy"]),
            "low_o3_rank": float(tr["low_o3_rank"]),
            "Low25_min_label": int(tr["Low25_min_label"]),
        }
        for col in ["NAM10", "NAM50", "NAM10_minus_NAM50"]:
            row[f"{col}_short_mean"] = window_mean(nam, col, short_start, cutoff_abs, 14)
            row[f"{col}_short_trend"] = window_trend(nam, col, short_start, cutoff_abs, 14)
        row["EP2_100hPa_long_mean"] = window_mean(ep, "EP2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["EP2_50hPa_long_mean"] = window_mean(ep, "EP2_50hPa_40_80N", long_start, cutoff_abs, 45)
        row["DIV2_100hPa_long_mean"] = window_mean(ep, "DIV2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["EP2_100hPa_short_mean"] = window_mean(ep, "EP2_100hPa_40_80N", short_start, cutoff_abs, 14)
        row["EP2_100hPa_long_trend"] = window_trend(ep, "EP2_100hPa_40_80N", long_start, cutoff_abs, 45)
        row["O3_short_mean"] = window_mean(o3, "O3_anom_rm5", short_start, cutoff_abs, 14)
        row["O3_short_trend"] = window_trend(o3, "O3_anom_rm5", short_start, cutoff_abs, 14)
        rows.append(row)
    features = pd.DataFrame(rows)
    all_predictors = sorted({x for xs in PREDICTOR_SETS.values() for x in xs})
    return features.dropna(subset=all_predictors + ["MA_O3_loss", "Low25_min_label"]).reset_index(drop=True)

example = build_features_for_cutoff(1, bundles["B2000WCN001002"])
example.head()


,dataset,year,cutoff_doy,cutoff_label,lead_days_to_Mar1,MA_O3_loss,MA_O3_min_DU,MA_O3_min_doy,low_o3_rank,Low25_min_label,NAM10_short_mean,NAM10_short_trend,NAM50_short_mean,NAM50_short_trend,NAM10_minus_NAM50_short_mean,NAM10_minus_NAM50_short_trend,EP2_100hPa_long_mean,EP2_50hPa_long_mean,DIV2_100hPa_long_mean,EP2_100hPa_short_mean,EP2_100hPa_long_trend,O3_short_mean,O3_short_trend
0,B2000WCN001002,2,1,01-01,59,8.793734,94.816771,90,38.0,1,-0.480374,-0.035064,-0.623432,0.066529,0.143058,-0.101592,-0.002469,-0.001709,-1.194364,-0.003658,-0.000045,-0.959425,0.104057
1,B2000WCN001002,3,1,01-01,59,20.850690,83.638120,88,5.0,1,2.892339,0.045442,2.285304,0.021372,0.607035,0.024071,-0.001552,-0.000878,-1.582300,-0.001364,-0.000006,-9.667860,-0.315071
2,B2000WCN001002,4,1,01-01,59,5.104077,97.625508,97,56.0,0,0.613153,0.021419,-0.101800,0.038889,0.714952,-0.017470,-0.002616,-0.001756,-1.383258,-0.002448,0.000003,2.459053,0.090193
3,B2000WCN001002,5,1,01-01,59,0.410272,98.540256,120,68.0,0,0.817157,0.101497,-0.870712,0.068927,1.687869,0.032570,-0.002408,-0.001405,-1.548550,-0.002130,0.000012,2.849851,-0.135389
4,B2000WCN001002,6,1,01-01,59,3.071182,101.140141,114,95.0,0,1.677662,0.129693,0.843744,0.117781,0.833918,0.011912,-0.002181,-0.001239,-1.728144,-0.002495,-0.000002,-2.497446,-0.139133


## Models and operational skill metrics

This block defines the ML algorithms and the scoring rules.

What it does:

- trains one classifier for the severe-event alarm (`Low25_min_label`),
- trains one regressor for seasonal loss magnitude (`MA_O3_loss`),
- trains one regressor for event depth (`MA_O3_min_DU`),
- evaluates all models with out-of-fold B2000WCN predictions.

Why these algorithms:

- `linear_l2`: transparent baseline with shrinkage, useful for checking signs and avoiding overfit.
- `random_forest`: shallow tree ensemble, useful for threshold-like behavior.
- `gradient_boosting`: shallow boosted trees, useful for weak nonlinear additive relationships.


In [6]:
MODEL_FAMILIES = ["linear_l2", "random_forest", "gradient_boosting"]

def make_regressor(model_family: str):
    if model_family == "linear_l2":
        return Pipeline([("scaler", StandardScaler()), ("reg", RidgeCV(alphas=np.logspace(-4, 4, 41)))])
    if model_family == "random_forest":
        return RandomForestRegressor(n_estimators=160, max_depth=4, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=2)
    if model_family == "gradient_boosting":
        return GradientBoostingRegressor(n_estimators=120, learning_rate=0.035, max_depth=2, min_samples_leaf=8, random_state=RANDOM_STATE)
    raise ValueError(model_family)

def make_classifier(model_family: str, y: np.ndarray):
    if model_family == "linear_l2":
        folds = min(5, int(np.bincount(y, minlength=2).min()))
        return Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegressionCV(Cs=np.logspace(-3, 3, 13), cv=max(3, folds), penalty="l2", solver="lbfgs", max_iter=5000, scoring="neg_brier_score", random_state=RANDOM_STATE))])
    if model_family == "random_forest":
        return RandomForestClassifier(n_estimators=160, max_depth=4, min_samples_leaf=5, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=2)
    if model_family == "gradient_boosting":
        return GradientBoostingClassifier(n_estimators=120, learning_rate=0.035, max_depth=2, min_samples_leaf=8, random_state=RANDOM_STATE)
    raise ValueError(model_family)

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    mse = float(mean_squared_error(y_true, y_pred))
    clim_mse = float(mean_squared_error(y_true, np.full_like(y_true, np.mean(y_true), dtype=float)))
    if len(y_true) >= 3 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        corr, pval = pearsonr(y_true, y_pred)
    else:
        corr, pval = np.nan, np.nan
    return {
        "reg_rmse": float(np.sqrt(mse)),
        "reg_mse_skill_vs_climatology": float(1 - mse / clim_mse) if clim_mse > 0 else np.nan,
        "reg_correlation": float(corr),
        "reg_correlation_pvalue": float(pval),
    }

def min_o3_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    metrics = regression_metrics(y_true, y_pred)
    return {
        "min_o3_rmse_DU": metrics["reg_rmse"],
        "min_o3_mse_skill_vs_climatology": metrics["reg_mse_skill_vs_climatology"],
        "min_o3_correlation": metrics["reg_correlation"],
        "min_o3_correlation_pvalue": metrics["reg_correlation_pvalue"],
    }

def top_fraction_alarm(y_true: np.ndarray, score: np.ndarray, fraction: float = EVENT_FRACTION) -> Dict[str, float]:
    valid = np.isfinite(score)
    y = y_true[valid].astype(int)
    s = score[valid].astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return {"top25_hit_rate": np.nan, "top25_false_alarm_rate": np.nan, "top25_precision": np.nan}
    alarm = np.zeros(len(y), dtype=int)
    n_alarm = max(1, int(math.floor(fraction * len(y))))
    alarm[np.argsort(-s)[:n_alarm]] = 1
    tp = int(((alarm == 1) & (y == 1)).sum())
    fp = int(((alarm == 1) & (y == 0)).sum())
    tn = int(((alarm == 0) & (y == 0)).sum())
    fn = int(((alarm == 0) & (y == 1)).sum())
    return {
        "top25_hit_rate": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "top25_false_alarm_rate": float(fp / (fp + tn)) if (fp + tn) else np.nan,
        "top25_precision": float(tp / (tp + fp)) if (tp + fp) else np.nan,
        "top25_tp": tp, "top25_fp": fp, "top25_tn": tn, "top25_fn": fn,
    }

def classification_metrics(y_true: np.ndarray, prob: np.ndarray) -> Dict[str, float]:
    valid = np.isfinite(prob)
    y = y_true[valid].astype(int)
    p = prob[valid].astype(float)
    if len(y) == 0 or len(np.unique(y)) < 2:
        out = {"cls_roc_auc": np.nan, "cls_brier_score": np.nan, "cls_brier_skill_vs_climatology": np.nan}
    else:
        brier = float(brier_score_loss(y, p))
        q = float(np.mean(y))
        brier_clim = q * (1 - q)
        out = {"cls_roc_auc": float(roc_auc_score(y, p)), "cls_brier_score": brier, "cls_brier_skill_vs_climatology": float(1 - brier / brier_clim) if brier_clim > 0 else np.nan}
    out.update(top_fraction_alarm(y_true, prob))
    return out

def evaluate_cv(df: pd.DataFrame, features: Sequence[str], model_family: str) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    sub = df.dropna(subset=list(features) + ["MA_O3_loss", "MA_O3_min_DU", "Low25_min_label"]).copy()
    x = sub[list(features)].to_numpy(dtype=float)
    y_loss = sub["MA_O3_loss"].to_numpy(dtype=float)
    y_min_o3 = sub["MA_O3_min_DU"].to_numpy(dtype=float)
    y_cls = sub["Low25_min_label"].to_numpy(dtype=int)
    cv_reg = KFold(n_splits=min(5, len(sub)), shuffle=True, random_state=RANDOM_STATE)
    loss_pred = cross_val_predict(make_regressor(model_family), x, y_loss, cv=cv_reg)
    min_o3_pred = cross_val_predict(make_regressor(model_family), x, y_min_o3, cv=cv_reg)
    prob = np.full(len(sub), np.nan)
    counts = np.bincount(y_cls, minlength=2)
    if counts.min() >= 3:
        cv_cls = StratifiedKFold(n_splits=min(5, int(counts.min())), shuffle=True, random_state=RANDOM_STATE)
        prob = cross_val_predict(make_classifier(model_family, y_cls), x, y_cls, cv=cv_cls, method="predict_proba")[:, 1]
    metrics = {
        **regression_metrics(y_loss, loss_pred),
        **min_o3_metrics(y_min_o3, min_o3_pred),
        **classification_metrics(y_cls, prob),
        "n_samples": int(len(sub)),
        "n_events": int(y_cls.sum()),
    }
    return metrics, loss_pred, min_o3_pred, prob

def fit_predict_external(train_df: pd.DataFrame, external_df: pd.DataFrame, features: Sequence[str], model_family: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    train = train_df.dropna(subset=list(features) + ["MA_O3_loss", "MA_O3_min_DU", "Low25_min_label"]).copy()
    ext = external_df.dropna(subset=list(features)).copy()
    x_train = train[list(features)].to_numpy(dtype=float)
    x_ext = ext[list(features)].to_numpy(dtype=float)
    y_loss = train["MA_O3_loss"].to_numpy(dtype=float)
    y_min_o3 = train["MA_O3_min_DU"].to_numpy(dtype=float)
    y_cls = train["Low25_min_label"].to_numpy(dtype=int)
    loss_reg = make_regressor(model_family)
    loss_reg.fit(x_train, y_loss)
    loss_pred = loss_reg.predict(x_ext)
    min_reg = make_regressor(model_family)
    min_reg.fit(x_train, y_min_o3)
    min_o3_pred = min_reg.predict(x_ext)
    prob = np.full(len(ext), np.nan)
    if np.bincount(y_cls, minlength=2).min() >= 3:
        clf = make_classifier(model_family, y_cls)
        clf.fit(x_train, y_cls)
        prob = clf.predict_proba(x_ext)[:, 1]
    return ext, loss_pred, min_o3_pred, prob


## Training block: lead-time scan

This is the main training block.

What it does:

1. For each cutoff date from 1 January to 28 February, build predictors using only information available by that date.
2. Train and evaluate models on B2000WCN using out-of-fold cross-validation.
3. Fit B2000WCN models and apply them to BWCN, so BWCN0008 can be inspected as an external severe-event case.

What it does **not** do:

- It does not save files; file writing is separated in the next output block.
- It does not use future O3 information after the cutoff date.


In [7]:
def add_probability_ranks(pred: pd.DataFrame) -> pd.DataFrame:
    out = pred.copy()
    out["prob_rank_desc"] = np.nan
    out["prob_top25_alarm"] = False
    group_cols = ["dataset", "cutoff_doy", "predictor_set", "model_family"]
    for _, idx in out.groupby(group_cols).groups.items():
        sub = out.loc[idx]
        valid_idx = sub[sub["pred_low25_probability"].notna()].index
        if len(valid_idx) == 0:
            continue
        ranks = sub.loc[valid_idx, "pred_low25_probability"].rank(method="first", ascending=False)
        n_alarm = max(1, int(math.floor(EVENT_FRACTION * len(valid_idx))))
        out.loc[valid_idx, "prob_rank_desc"] = ranks
        out.loc[valid_idx, "prob_top25_alarm"] = ranks <= n_alarm
    return out

def is_usable(row: pd.Series) -> bool:
    return bool(
        row.get("cls_roc_auc", np.nan) >= 0.75
        and row.get("cls_brier_skill_vs_climatology", np.nan) > 0
        and row.get("top25_hit_rate", np.nan) >= 0.60
        and row.get("top25_false_alarm_rate", np.nan) <= 0.35
        and row.get("reg_correlation", np.nan) >= 0.50
        and row.get("reg_mse_skill_vs_climatology", np.nan) > 0
    )

def sustained_earliest_lead(sub: pd.DataFrame, streak: int = 7) -> float:
    ordered = sub.sort_values("lead_days_to_Mar1", ascending=False).reset_index(drop=True)
    usable = ordered["usable_predictability"].to_numpy(dtype=bool)
    leads = ordered["lead_days_to_Mar1"].to_numpy(dtype=int)
    for i in range(len(ordered) - streak + 1):
        if usable[i : i + streak].all():
            return float(leads[i])
    return np.nan

def summarize_windows(summary: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, pset), sub in summary.groupby(["model_family", "predictor_set"]):
        usable = sub[sub["usable_predictability"]]
        rows.append({
            "model_family": model,
            "predictor_set": pset,
            "earliest_single_usable_lead_days": float(usable["lead_days_to_Mar1"].max()) if not usable.empty else np.nan,
            "earliest_7day_stable_usable_lead_days": sustained_earliest_lead(sub, 7),
            "best_auc": float(sub["cls_roc_auc"].max()),
            "best_loss_correlation": float(sub["reg_correlation"].max()),
            "best_loss_mse_skill": float(sub["reg_mse_skill_vs_climatology"].max()),
            "best_min_o3_correlation": float(sub["min_o3_correlation"].max()),
            "best_min_o3_mse_skill": float(sub["min_o3_mse_skill_vs_climatology"].max()),
        })
    return pd.DataFrame(rows).sort_values(["earliest_7day_stable_usable_lead_days", "earliest_single_usable_lead_days", "best_auc"], ascending=False, na_position="last")

train_bundle = bundles["B2000WCN001002"]
bwcn_bundle = bundles["BWCN"]
summary_rows = []
external_rows = []

for cutoff_doy in CUTOFF_DOYS:
    label = cutoff_label(cutoff_doy)
    print(f"[scan] {label}, lead={TARGET_START_DOY - cutoff_doy} d", flush=True)
    train_features = build_features_for_cutoff(cutoff_doy, train_bundle)
    bwcn_features = build_features_for_cutoff(cutoff_doy, bwcn_bundle)
    for predictor_set, features in PREDICTOR_SETS.items():
        for model_family in MODEL_FAMILIES:
            metrics, _, _, _ = evaluate_cv(train_features, features, model_family)
            summary_rows.append({
                "dataset": "B2000WCN001002",
                "cutoff_doy": cutoff_doy,
                "cutoff_label": label,
                "lead_days_to_Mar1": TARGET_START_DOY - cutoff_doy,
                "predictor_set": predictor_set,
                "model_family": model_family,
                "features": ",".join(features),
                **metrics,
            })
            ext, ext_loss, ext_min_o3, ext_prob = fit_predict_external(train_features, bwcn_features, features, model_family)
            keep = ext[["dataset", "year", "cutoff_doy", "cutoff_label", "lead_days_to_Mar1", "MA_O3_loss", "MA_O3_min_DU", "MA_O3_min_doy", "low_o3_rank", "Low25_min_label"]].copy()
            keep["predictor_set"] = predictor_set
            keep["model_family"] = model_family
            keep["pred_MA_O3_loss"] = ext_loss
            keep["pred_MA_O3_min_DU"] = ext_min_o3
            keep["pred_low25_probability"] = ext_prob
            external_rows.append(keep)

skill_summary = pd.DataFrame(summary_rows)
skill_summary["usable_predictability"] = skill_summary.apply(is_usable, axis=1)
window_summary = summarize_windows(skill_summary)
external_predictions = add_probability_ranks(pd.concat(external_rows, ignore_index=True))
bwcn0008_trace = external_predictions[external_predictions["year"].astype(int) == 8].copy()

window_summary


[scan] 01-01, lead=59 d


[scan] 01-02, lead=58 d


[scan] 01-03, lead=57 d


[scan] 01-04, lead=56 d


[scan] 01-05, lead=55 d


...

[scan] 02-25, lead=4 d


[scan] 02-26, lead=3 d


[scan] 02-27, lead=2 d


[scan] 02-28, lead=1 d


,model_family,predictor_set,earliest_single_usable_lead_days,earliest_7day_stable_usable_lead_days,best_auc,best_loss_correlation,best_loss_mse_skill,best_min_o3_correlation,best_min_o3_mse_skill
9,random_forest,DYN_PLUS_PASSIVE_TRACER,22.0,15.0,0.874937,0.827345,0.684413,0.716199,0.512445
1,gradient_boosting,DYN_PLUS_PASSIVE_TRACER,20.0,15.0,0.878079,0.824658,0.679045,0.711829,0.506055
5,linear_l2,DYN_PLUS_PASSIVE_TRACER,15.0,15.0,0.902966,0.857916,0.735963,0.724543,0.524904
0,gradient_boosting,DYN_COUPLED,NaN,NaN,0.813725,0.688162,0.472430,0.582635,0.339180
8,random_forest,DYN_COUPLED,NaN,NaN,0.785714,0.664674,0.441441,0.578274,0.334240
2,gradient_boosting,VORTEX_STATE,NaN,NaN,0.784759,0.629576,0.396313,0.554240,0.307139
4,linear_l2,DYN_COUPLED,NaN,NaN,0.783677,0.696482,0.484386,0.592815,0.351000
10,random_forest,VORTEX_STATE,NaN,NaN,0.783359,0.604424,0.360786,0.537873,0.285079
3,gradient_boosting,WAVE_FORCING,NaN,NaN,0.781767,0.626206,0.391991,0.527363,0.275185
6,linear_l2,VORTEX_STATE,NaN,NaN,0.777401,0.608609,0.370276,0.534048,0.285103


## Output block: save compact training outputs

This block writes the compact outputs from the training scan.

Saved objects:

- `leadtime_skill_summary.csv`: B2000WCN out-of-fold skill at each cutoff, predictor group, and model.
- `predictability_windows.csv`: the distilled lead-time answer for each predictor group and model.
- `bwcn0008_trace.csv`: BWCN0008 transfer predictions through lead time.

This separation keeps training logic and file-writing logic easy to audit.


In [8]:
skill_summary.to_csv(OUT_DIR / "leadtime_skill_summary.csv", index=False)
window_summary.to_csv(OUT_DIR / "predictability_windows.csv", index=False)
bwcn0008_trace.to_csv(OUT_DIR / "bwcn0008_trace.csv", index=False)

for name in ["leadtime_skill_summary.csv", "predictability_windows.csv", "bwcn0008_trace.csv"]:
    path = OUT_DIR / name
    print(f"saved {path} ({path.stat().st_size} bytes)")


saved /home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/leadtime_skill_summary.csv (359982 bytes)
saved /home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/predictability_windows.csv (1737 bytes)
saved /home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/bwcn0008_trace.csv (112419 bytes)


## External output block: BWCN and BWCN0008

This block summarizes the BWCN transfer test.

Scientific purpose:

- Check whether B2000WCN-trained dynamical relationships also rank severe BWCN events highly.
- Inspect BWCN0008, the very severe event, as a concrete case study.
- Report both event-alarm skill and intensity skill, including predicted minimum O3 depth.


In [9]:
def best_external_summary(pred: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, pset), sub in pred.groupby(["model_family", "predictor_set"]):
        metric_rows = []
        for _, g in sub.groupby("cutoff_doy"):
            y = g["Low25_min_label"].to_numpy(dtype=int)
            prob = g["pred_low25_probability"].to_numpy(dtype=float)
            loss = regression_metrics(g["MA_O3_loss"].to_numpy(dtype=float), g["pred_MA_O3_loss"].to_numpy(dtype=float))
            depth = min_o3_metrics(g["MA_O3_min_DU"].to_numpy(dtype=float), g["pred_MA_O3_min_DU"].to_numpy(dtype=float))
            cls = classification_metrics(y, prob)
            metric_rows.append({**loss, **depth, **cls})
        m = pd.DataFrame(metric_rows)
        rows.append({
            "model_family": model,
            "predictor_set": pset,
            "best_auc": float(m["cls_roc_auc"].max()),
            "best_loss_correlation": float(m["reg_correlation"].max()),
            "best_min_o3_correlation": float(m["min_o3_correlation"].max()),
            "best_top25_hit_rate": float(m["top25_hit_rate"].max()),
            "best_top25_false_alarm_rate": float(m["top25_false_alarm_rate"].min()),
        })
    return pd.DataFrame(rows).sort_values(["best_auc", "best_loss_correlation"], ascending=False)

external_summary = best_external_summary(external_predictions)
external_summary.to_csv(OUT_DIR / "bwcn_external_skill_summary.csv", index=False)

bwcn0008_alarm = (
    bwcn0008_trace[bwcn0008_trace["prob_top25_alarm"]]
    .groupby(["model_family", "predictor_set"])["lead_days_to_Mar1"]
    .max()
    .reset_index(name="earliest_top25_alarm_lead_days")
    .sort_values("earliest_top25_alarm_lead_days", ascending=False)
)
bwcn0008_alarm.to_csv(OUT_DIR / "bwcn0008_alarm_summary.csv", index=False)

external_summary


,model_family,predictor_set,best_auc,best_loss_correlation,best_min_o3_correlation,best_top25_hit_rate,best_top25_false_alarm_rate
1,gradient_boosting,DYN_PLUS_PASSIVE_TRACER,1.000000,0.951173,0.925934,1.0,0.000000
5,linear_l2,DYN_PLUS_PASSIVE_TRACER,1.000000,0.928002,0.873903,1.0,0.000000
10,random_forest,VORTEX_STATE,0.988889,0.923010,0.903946,0.8,0.055556
0,gradient_boosting,DYN_COUPLED,0.988889,0.886314,0.851571,0.8,0.055556
4,linear_l2,DYN_COUPLED,0.988889,0.863460,0.815135,0.8,0.055556
11,random_forest,WAVE_FORCING,0.988889,0.850982,0.809954,0.8,0.055556
9,random_forest,DYN_PLUS_PASSIVE_TRACER,0.988235,0.953912,0.923682,0.8,0.055556
7,linear_l2,WAVE_FORCING,0.988235,0.833449,0.798251,0.8,0.055556
2,gradient_boosting,VORTEX_STATE,0.977778,0.912544,0.901466,0.8,0.055556
3,gradient_boosting,WAVE_FORCING,0.976471,0.861140,0.810247,0.8,0.055556


## Plot block: what each figure answers

This block creates four figures. Each figure has a specific scientific question.

1. `fig_leadtime_auc.png`
   - Object plotted: lead time versus severe-event AUC.
   - Purpose: ask **how early each dynamical predictor group can distinguish severe low-O3 years from other years**.

2. `fig_leadtime_reg_correlation.png`
   - Object plotted: lead time versus out-of-fold correlation for `MA_O3_loss`.
   - Purpose: ask **how early each group can estimate the seasonal ozone-loss magnitude**.

3. `fig_leadtime_min_o3_correlation.png`
   - Object plotted: lead time versus out-of-fold correlation for `MA_O3_min_DU`.
   - Purpose: ask **how early each group can estimate how low the event roughly gets**.

4. `fig_bwcn0008_probability.png`
   - Object plotted: BWCN0008 predicted low-O3 probability versus lead time.
   - Purpose: ask **whether the severe BWCN0008 case would be flagged early by B2000WCN-trained dynamical models**.


In [10]:
def plot_leadtime(summary: pd.DataFrame, metric: str, ylabel: str, outfile: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=False)
    axes = axes.ravel()
    for ax, predictor_set in zip(axes, PREDICTOR_SETS):
        sub0 = summary[summary["predictor_set"] == predictor_set]
        for model, sub in sub0.groupby("model_family"):
            sub = sub.sort_values("lead_days_to_Mar1")
            ax.plot(sub["lead_days_to_Mar1"], sub[metric], marker="o", markersize=3, linewidth=1.2, label=model)
        ax.set_title(predictor_set)
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("Lead days before 1 March")
        ax.set_ylabel(ylabel)
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile, dpi=180)
    plt.close(fig)

def plot_bwcn0008(trace: pd.DataFrame, outfile: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
    axes = axes.ravel()
    for ax, predictor_set in zip(axes, PREDICTOR_SETS):
        sub0 = trace[trace["predictor_set"] == predictor_set]
        for model, sub in sub0.groupby("model_family"):
            sub = sub.sort_values("lead_days_to_Mar1")
            ax.plot(sub["lead_days_to_Mar1"], sub["pred_low25_probability"], marker="o", markersize=3, linewidth=1.2, label=model)
        ax.set_title(predictor_set)
        ax.grid(True, alpha=0.25)
        ax.set_xlabel("Lead days before 1 March")
        ax.set_ylabel("BWCN0008 low-O3 probability")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile, dpi=180)
    plt.close(fig)

plot_leadtime(skill_summary, "cls_roc_auc", "Low-O3 event AUC", OUT_DIR / "fig_leadtime_auc.png")
plot_leadtime(skill_summary, "reg_correlation", "MA ozone-loss correlation", OUT_DIR / "fig_leadtime_reg_correlation.png")
plot_leadtime(skill_summary, "min_o3_correlation", "Minimum-O3 depth correlation", OUT_DIR / "fig_leadtime_min_o3_correlation.png")
plot_bwcn0008(bwcn0008_trace, OUT_DIR / "fig_bwcn0008_probability.png")

for png in ["fig_leadtime_auc.png", "fig_leadtime_reg_correlation.png", "fig_leadtime_min_o3_correlation.png", "fig_bwcn0008_probability.png"]:
    print(OUT_DIR / png)


/home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/fig_leadtime_auc.png
/home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/fig_leadtime_reg_correlation.png
/home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/fig_leadtime_min_o3_correlation.png
/home/weiji/restart_exam/code_cleaned/ML/outputs/dynamical_predictability/fig_bwcn0008_probability.png


## Report block: write the plain-language findings

This final output block writes a short Markdown report. The report explicitly separates:

- the research question,
- the operational definition of predictability,
- event predictability results,
- intensity predictability results,
- the BWCN0008 case-study interpretation.


In [11]:
def fmt(value, digits=3) -> str:
    if value is None or not np.isfinite(value):
        return "nan"
    return f"{float(value):.{digits}f}"

def markdown_table(df: pd.DataFrame, columns: Optional[Sequence[str]] = None) -> str:
    frame = df.loc[:, list(columns)].copy() if columns is not None else df.copy()
    if frame.empty:
        return "_No rows._"
    def cell(value) -> str:
        if isinstance(value, (float, np.floating)):
            return fmt(float(value))
        return str(value)
    headers = list(frame.columns)
    rows = [[cell(v) for v in row] for row in frame.to_numpy()]
    widths = [max(len(str(h)), *(len(row[i]) for row in rows)) for i, h in enumerate(headers)]
    lines = ["| " + " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)) + " |",
             "| " + " | ".join("-" * widths[i] for i in range(len(headers))) + " |"]
    lines.extend("| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(headers))) + " |" for row in rows)
    return "\n".join(lines)

report = []
report.append("# Dynamical ozone-loss predictability scan")
report.append("")
report.append("## Research question")
report.append("")
report.append("How early can winter stratospheric dynamics identify and quantify severe Arctic ozone-loss years in the longrun?")
report.append("")
report.append("## Predictability definition")
report.append("")
report.append("Predictability is defined in two complementary ways. Event predictability asks whether a year enters the lowest 25% of March-April minimum polar-cap O3. Intensity predictability asks how severe the year is, using both March-April mean ozone-loss anomaly (`MA_O3_loss`) and the annual March-April minimum O3 depth (`MA_O3_min_DU`).")
report.append("")
report.append("So the answer is not only a low-25% classification, and it is not a full daily O3 trajectory simulation. The notebook tests whether dynamics can provide an early alarm plus an approximate severity estimate.")
report.append("")
report.append("The stable lead-time gate uses event skill plus seasonal-loss regression skill. Minimum-O3 depth is reported separately because a single-day minimum is noisier, but it directly answers whether we can estimate roughly how low the ozone gets.")
report.append("")
report.append("O3 is not evaluated as a standalone predictor set. When included, it is treated as a passive tracer inside `DYN_PLUS_PASSIVE_TRACER`.")
report.append("")
report.append("## Stable B2000WCN lead windows")
report.append("")
report.append(markdown_table(window_summary))
report.append("")
report.append("## BWCN external check")
report.append("")
report.append(markdown_table(external_summary, ["model_family", "predictor_set", "best_auc", "best_loss_correlation", "best_min_o3_correlation", "best_top25_hit_rate", "best_top25_false_alarm_rate"]))
report.append("")
report.append("## BWCN0008 top-quartile alarms")
report.append("")
report.append(markdown_table(bwcn0008_alarm))
report.append("")
report.append("## Figure guide")
report.append("")
report.append("- `fig_leadtime_auc.png`: lead time versus severe-event AUC; answers when low-O3 events become separable.")
report.append("- `fig_leadtime_reg_correlation.png`: lead time versus `MA_O3_loss` correlation; answers when seasonal loss magnitude becomes estimable.")
report.append("- `fig_leadtime_min_o3_correlation.png`: lead time versus `MA_O3_min_DU` correlation; answers when the approximate event depth becomes estimable.")
report.append("- `fig_bwcn0008_probability.png`: BWCN0008 event probability versus lead time; answers whether a known severe case is flagged early.")
report.append("")
report.append("## Interpretation")
report.append("")
report.append("- The vortex-only and wave-only partitions test whether the early signal sits primarily in the vortex state or in wave forcing.")
report.append("- `DYN_COUPLED` tests the physical pathway in which wave forcing modulates the vortex and therefore later ozone loss.")
report.append("- `DYN_PLUS_PASSIVE_TRACER` tests whether ozone as a passive tracer of prior transport/chemistry sharpens alarms; it should not be interpreted as a separate non-dynamical source of predictability.")
report.append("- True vortex morphology is not yet diagnosed here. NAM vertical coupling is only a compact proxy; a next step would compute vortex moment diagnostics from geopotential height or PV-like fields.")

(OUT_DIR / "findings.md").write_text("\n".join(report) + "\n")
print((OUT_DIR / "findings.md").read_text())


# Dynamical ozone-loss predictability scan

## Research question

How early can winter stratospheric dynamics identify and quantify severe Arctic ozone-loss years in the longrun?

## Predictability definition

Predictability is defined in two complementary ways. Event predictability asks whether a year enters the lowest 25% of March-April minimum polar-cap O3. Intensity predictability asks how severe the year is, using both March-April mean ozone-loss anomaly (`MA_O3_loss`) and the annual March-April minimum O3 depth (`MA_O3_min_DU`).

So the answer is not only a low-25% classification, and it is not a full daily O3 trajectory simulation. The notebook tests whether dynamics can provide an early alarm plus an approximate severity estimate.

The stable lead-time gate uses event skill plus seasonal-loss regression skill. Minimum-O3 depth is reported separately because a single-day minimum is noisier, but it directly answers whether we can estimate roughly how low the ozone gets.

O3 is no